In [ ]:
def is_iterable(data):
    return isinstance(data, list)

def flatten(data):
    if not is_iterable(data):
        return [data]

    out = []
    for item in data:
        out.extend(flatten(item))
    return out

def get_shape(data):
    if not is_iterable(data):
        return ()

    assert data, "empty tensors skipped for now"

    inner = get_shape(data[0])
    
    for item in data:
        assert get_shape(item) == inner, "ragged tensor"

    return (len(data),) + inner

def get_strides_from_shape(shape):
    strides = []
    running = 1
    for size in reversed(shape):
        strides.append(running)
        running *= size
    return tuple(reversed(strides))

def index_to_position(index, strides):
    return sum(i * stride for i, stride in zip(index * strides))

class Tensor:

    def __init__(
        self,
        data,
        requires_grad: bool,
        children: tuple[Tensor] = (),
    ):
        self._requires_grad = requires_grad

        self._storage = flatten(data)
        self._shape = get_shape(data)
        self._strides = get_strides_from_shape(self._shape)

        self._grad = [0.0 for _ in data]
        self._children = children

    def __mul__(self, other) -> Tensor:
        raise NotImplementedError
    
    def sum(self) -> Tensor:
        raise NotImplementedError

    # and a bunch of other operations like addition and subtraction etc

    def backward(self) -> None:
        raise NotImplementedError

In [ ]:
x = Tensor([[1, 2, 3],
            [4, 5, 6]], requires_grad=True)   # shape (2, 3)

w = Tensor([10, 20, 30], requires_grad=True)   # shape (3,)

y = x * w                                     # broadcast w -> shape (2, 3)
loss = y.sum()                                # scalar
loss.backward()